In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.formula_1.silver_fact_laps;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.formula_1.silver_fact_laps (
  session_key INT,
  meeting_key INT,
  driver_number INT,
  lap_number INT,
  lap_start_time TIMESTAMP,
  lap_duration_seconds DOUBLE,
  s1_seconds DOUBLE,
  s2_seconds DOUBLE,
  s3_seconds DOUBLE,
  speed_trap_1 INT,
  speed_trap_2 INT,
  top_speed INT,
  is_pit_out_lap BOOLEAN,
  is_driver_personal_best BOOLEAN,
  year INT,
  month INT
)
USING DELTA;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT session_key) AS total_sessions,
    COUNT(DISTINCT meeting_key) AS total_meetings,
    COUNT(DISTINCT driver_number) AS total_drivers,
    COUNT(DISTINCT lap_number) AS total_laps,
    MIN(lap_start_time) AS min_date,
    MAX(lap_start_time) AS max_date
FROM
    workspace.formula_1.silver_fact_laps

In [0]:
%sql
MERGE INTO workspace.formula_1.silver_fact_laps AS target
USING (
  WITH parsed_laps AS (
    SELECT 
      year,
      month,
      from_json(raw_json, 'lap_number INT, meeting_key INT, session_key INT, driver_number INT, lap_duration DOUBLE, duration_sector_1 DOUBLE, duration_sector_2 DOUBLE, duration_sector_3 DOUBLE, i1_speed INT, i2_speed INT, st_speed INT, is_pit_out_lap BOOLEAN, date_start STRING') AS l
    FROM workspace.formula_1.bronze_laps
  )
  SELECT 
    l.session_key,
    l.meeting_key,
    l.driver_number,
    l.lap_number,
    to_timestamp(l.date_start) AS lap_start_time,
    l.lap_duration AS lap_duration_seconds,
    l.duration_sector_1 AS s1_seconds,
    l.duration_sector_2 AS s2_seconds,
    l.duration_sector_3 AS s3_seconds,
    l.i1_speed AS speed_trap_1,
    l.i2_speed AS speed_trap_2,
    l.st_speed AS top_speed,
    l.is_pit_out_lap,
    CASE 
      WHEN l.lap_duration = MIN(l.lap_duration) OVER(PARTITION BY l.session_key, l.driver_number) THEN true 
      ELSE false 
    END AS is_driver_personal_best,
    l.year,
    l.month
  FROM parsed_laps l
  WHERE l.lap_duration IS NOT NULL 
    AND l.lap_duration > 50.0
    AND l.lap_duration < 300.0
) AS source
ON target.session_key = source.session_key 
   AND target.driver_number = source.driver_number
   AND target.lap_number = source.lap_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT session_key) AS total_sessions,
    COUNT(DISTINCT meeting_key) AS total_meetings,
    COUNT(DISTINCT driver_number) AS total_drivers,
    COUNT(DISTINCT lap_number) AS total_laps,
    MIN(lap_start_time) AS min_date,
    MAX(lap_start_time) AS max_date
FROM
    workspace.formula_1.silver_fact_laps

In [0]:
%sql
SELECT
*
FROM
workspace.formula_1.silver_fact_laps
ORDER BY lap_start_time DESC
LIMIT 5;